- 1.	create()              -> Create new table only
- 2.	createIfNotExists()   -> Create only if table doesn't exist
- 3.	createOrReplace()     -> Create new or replace existing table


# ### python

# Method 1: create()
# Creates a new Delta table. Fails if the table already exists.

In [0]:
from delta.tables import *
DeltaTable.create(spark) \
    .tableName("employee_demo") \
    .addColumn("emp_id", "INT") \
    .addColumn("emp_name", "STRING") \
    .addColumn("gender", "STRING") \
    .addColumn("salary", "INT") \
    .addColumn("Dept", "STRING") \
    .property("description", "table created for demo purpose") \
    .location("/FileStore/tables/delta/createtable") \
    .execute()


# Method 2: createIfNotExists()
# Creates the table only if it does not already exist.


In [0]:
from delta.tables import *
DeltaTable.createIfNotExists(spark) \
    .tableName("employee_demo") \
    .addColumn("emp_id", "INT") \
    .addColumn("emp_name", "STRING") \
    .addColumn("gender", "STRING") \
    .addColumn("salary", "INT") \
    .addColumn("Dept", "STRING") \
    .execute()


# Method 3: createOrReplace()
# Creates the table if it doesn't exist; otherwise replaces the existing table definition.
# 


In [0]:
from delta.tables import *
DeltaTable.createOrReplace(spark) \
    .tableName("tri.tri.employee_demo") \
    .addColumn("emp_id", "INT") \
    .addColumn("emp_name", "STRING") \
    .addColumn("gender", "STRING") \
    .addColumn("salary", "INT") \
    .addColumn("Dept", "STRING") \
    .property("description", "table created for demo purpose") \
    .execute()



# Bonus Method 4: External Table in Unity Catalog


- •	Data is stored in a user-specified location. 
- •	Table metadata is stored in Unity Catalog. 
- •	Commonly used with ADLS Gen2 external locations.


In [0]:
from delta.tables import *
DeltaTable.createIfNotExists(spark) \
    .tableName("main.default.employee_demo") \
    .addColumn("emp_id", "INT") \
    .addColumn("emp_name", "STRING") \
    .addColumn("gender", "STRING") \
    .addColumn("salary", "INT") \
    .addColumn("dept", "STRING") \
    .location("abfss://container@storageaccount.dfs.core.windows.net/delta/employee_demo") \
    .execute()


**Note:** In previous Databricks versions, DBFS was available and commonly used. In the current environment, DBFS Root is disabled, and Unity Catalog Volumes are the recommended storage option. Volumes work directly with SQL using the /Volumes/<catalog>/<schema>/<volume>/ path. If you want to access the same storage from Python APIs that require a cloud storage URI, use the underlying container-level path (for example, an abfss://, s3://, or gs:// URL) instead of the /Volumes/... path.

# sql

# Method 1: CREATE TABLE
# •	Creates a managed Delta table. 
# •	Fails if the table already exists.
# 

In [0]:
%sql
CREATE TABLE tri.tri.employee_demo (
    emp_id INT,
    emp_name STRING,
    gender STRING,
    salary INT,
    dept STRING
)
USING DELTA;


# Method 2: CREATE TABLE IF NOT EXISTS
# •	Creates the table only when it doesn't exist. 
# •	No error if the table already exists.
# 

In [0]:
%sql
CREATE TABLE IF NOT EXISTS tri.tri.employee_demo (
    emp_id INT,
    emp_name STRING,
    gender STRING,
    salary INT,
    dept STRING
)
USING DELTA;


# Method 3: CREATE OR REPLACE TABLE
# •	Creates the table if it doesn't exist. 
# •	Replaces the existing table definition if it exists.


In [0]:
%sql
CREATE OR REPLACE TABLE tri.tri.employee_demo (
    emp_id INT,
    emp_name STRING,
    gender STRING,
    salary INT,
    dept STRING
)
USING DELTA;



# Method 4: CTAS (Create Table As Select)
# •	Creates the table and loads data in one step. 
# •	Schema is automatically inferred from the SELECT statement.
# 

In [0]:
%sql
CREATE TABLE tri.tri.employee_demo
USING DELTA
AS
SELECT
    1 AS emp_id,
    'John' AS emp_name,
    'Male' AS gender,
    50000 AS salary,
    'IT' AS dept;


# Bonus Method 5: External Table in Unity Catalog
# •	Data is stored in a user-specified location. 
# •	Table metadata is stored in Unity Catalog. 
# •	Commonly used with ADLS Gen2 external locations.


In [0]:
%sql
CREATE TABLE tri.tri.employee_demo (
    emp_id INT,
    emp_name STRING,
    gender STRING,
    salary INT,
    dept STRING
)
USING DELTA
LOCATION 'abfss://container@storageaccount.dfs.core.windows.net/delta/employee_demo';


# Using DataFrame (Unity Catalog)


In [0]:

# Step 1: Create DataFrame
employee_data = [
    (100, "Stephen", "M", 2100, "IT"),
    (200, "Philipp", "M", 8000, "HR"),
    (50000, "Lara", "F", 6000, "SALES")
]
employee_schema = "emp_id INT, emp_name STRING, gender STRING, salary INT, dept STRING"
df = spark.createDataFrame(
    data=employee_data,
    schema=employee_schema
)
display(df)
# Step 2: Save as Unity Catalog Managed Table
df.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable("tri.tri.employee_demo")



In [0]:
%sql
SELECT COUNT(*) FROM tri.tri.employee_demo;

In [0]:
%sql
describe  tri.tri.employee_demo